# Copy a Globus collection into a Lifebit Jupyter session

**Why this notebook exists.** NIH HPC collections, such as `71c2ad2f-ea02-4d0e-8408-ac50f5af0f7d`, have HTTPS download switched off on purpose, for security. Downloading by HTTPS answers HTTP 500 ("globus_xio: System error in connect"). NIH HPC staff advise installing Globus Connect Personal, creating your own endpoint and transferring to it. This notebook does that: your Jupyter session becomes a Globus endpoint and Globus Transfer copies the files into it.

**How to read it.** Each step is a short code cell followed by a plain explanation, in the style of literate programming (Knuth, 1984; Rule et al., 2019). Run the cells from top to bottom. Step 4 is done in a Terminal, not in the notebook.

**Before you start:** request a session with enough disk (about 600 GB for `BW_111226_TOFA`, about 1.2 TB for everything, not yet measured) and have your `globus_tokens` text (the base64 text from the login step) ready to paste.

In [ ]:
SOURCE = "71c2ad2f-ea02-4d0e-8408-ac50f5af0f7d"   # the NIH collection
SOURCE_PATH = "/BW_111226_TOFA/"                  # folder to copy (start small, see step 7)
DEST_DIR = "/home/jovyan/globus_data"             # where files land in this session

**Step 1. Settings.** These three lines are the only things you change. `SOURCE` is the Globus collection, `SOURCE_PATH` the folder in it, and `DEST_DIR` the folder in this session that will receive the files. Check that `DEST_DIR` is on the big disk (step 2 shows the disks); change it if not.

In [ ]:
!df -h
!mkdir -p {DEST_DIR}
!pip install -q globus-sdk requests

**Step 2. Disk and packages.** `df -h` lists the disks and their free space. Find the one holding `DEST_DIR` and compare its free space with the size of what you will copy. The Globus Python package is installed so the notebook can talk to Globus.

In [ ]:
import getpass, os, sys
sys.path.insert(0, "../container/globus_tools")
os.environ["GLOBUS_TOKENS"] = getpass.getpass("Paste globus_tokens, then press Enter: ")

**Step 3. Your login.** Paste the base64 text from the login step. It is typed into a hidden prompt, so it is not shown on screen or saved in the notebook. It lives only in this session's memory. The notebook assumes you opened it from the repository's `ipynb/` folder, so the path `../container/globus_tools` finds our `auth.py`.

**Step 4. Install and set up Globus Connect Personal (in a Terminal).** In JupyterLab choose File > New > Terminal and type these lines, one at a time:

```bash
cd ~
wget https://downloads.globus.org/globus-connect-personal/linux/stable/globusconnectpersonal-latest.tgz
tar xzf globusconnectpersonal-latest.tgz
cd globusconnectpersonal-*
./globusconnectpersonal -setup
```

The setup asks for text input. Follow what it prints; I have not seen its exact prompts, so read them. It will likely show a web address to open in your browser, where you log in with your PIV card and get a one-time code to paste back. Give the endpoint a name such as `lifebit-session`. This follows the [Globus Connect Personal Linux guide](https://docs.globus.org/globus-connect-personal/install/linux/).

Then tell it which folder may receive files. The default only allows your home folder. Replace `/home/jovyan/globus_data` with your `DEST_DIR`:

```bash
echo "/home/jovyan/globus_data,0,1" > ~/.globusonline/lta/config-paths
```

The line means: path, sharing off (0), read and write on (1).

In [ ]:
import subprocess
subprocess.Popen("nohup ./globusconnectpersonal -start > ~/gcp.log 2>&1 &",
                 shell=True, cwd=os.path.expanduser("~") + "/" + [d for d in os.listdir(os.path.expanduser("~")) if d.startswith("globusconnectpersonal-") and os.path.isdir(os.path.expanduser("~/" + d))][0])

**Step 5. Start the endpoint.** This starts Globus Connect Personal in the background so it keeps running while the notebook does other things. If you start it a second time nothing bad happens. The log is `~/gcp.log`.

In [ ]:
import time; time.sleep(10)
!cd ~/globusconnectpersonal-* && ./globusconnectpersonal -status
!pip install -q globus-cli
DEST = !globus endpoint local-id
DEST = DEST[0].strip()
print("destination endpoint:", DEST)

**Step 6. Check it is online and get its id.** `-status` should say `Globus Online: connected`. If it does not, wait a minute and run the cell again. The id printed at the end is the destination for the transfer. The `local-id` command is the one the Globus guide gives for this.

In [ ]:
import globus_list, auth
client = auth.transfer_client()
files = globus_list.list_files(client, SOURCE, SOURCE_PATH)
print(len(files), "files,", round(sum(f["size"] for f in files) / 1e9, 1), "GB")

**Step 7. How much will move.** This lists everything under `SOURCE_PATH` with the same code the workflow uses and prints the file count and total size. Compare the total with the free disk from step 2. If it does not fit, choose a smaller `SOURCE_PATH`.

In [ ]:
!python ../container/globus_tools/globus_transfer.py --file --source {SOURCE} --dest {DEST} --source-path /fastq_md5sum.txt --dest-path {DEST_DIR}/fastq_md5sum.txt

**Step 8. Test with one small file.** This sends only the 40 KB checksum file at the top of the collection. It proves the whole chain works (login, endpoint, permission to write) before 600 GB is committed. The first time, Globus may ask for consent for the destination. If the task stops as `INACTIVE`, read its message; it names the missing permission.

In [ ]:
!python ../container/globus_tools/globus_transfer.py --source {SOURCE} --dest {DEST} --source-path {SOURCE_PATH} --dest-path {DEST_DIR}{SOURCE_PATH}

**Step 9. Copy the folder.** Globus Transfer copies every file, retries any that fail, and compares checksums. The cell prints progress every 30 seconds. The transfer is run by Globus, so it keeps going even if you close the browser, but the session itself must stay alive. Time is about size divided by speed; measure the speed on the first big file before you plan around it.

In [ ]:
!cd {DEST_DIR}{SOURCE_PATH} && md5sum -c fastq_md5sum.txt | grep -v ': OK$' || echo "all files match"

**Step 10. Check the files against the md5 list.** `md5sum -c` recomputes each checksum and compares it with the list that came with the data. Anything not OK is printed; if nothing is printed you see `all files match`. This never deletes anything.

**Step 11. Finish.** Copy the files from `DEST_DIR` into your Lifebit project storage, then stop and remove the endpoint in the Terminal:

```bash
cd ~/globusconnectpersonal-*
./globusconnectpersonal -stop
```

Globus's guide says to delete the install folder and `~/.globusonline/` to remove the software. It does not say how to remove the endpoint from your Globus account, so delete it by hand in the Globus web app (Collections) if you do not want it listed.

**References.** Knuth, D. E. (1984) Literate Programming. *The Computer Journal* 27(2):97-111. Rule, A., Tabard, A. and Hollan, J. (2019) Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks. *PLOS Computational Biology* 15(7):e1007007.